# TASK 1 Reproduce LLMOPT on a Standard Benchmark

In [1]:
!nvidia-smi

Tue Jun  2 07:08:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cloning the Repository

In [2]:
!git clone https://github.com/caigaojiang/LLMOPT

Cloning into 'LLMOPT'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 141 (delta 58), reused 46 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.47 MiB | 5.32 MiB/s, done.
Resolving deltas: 100% (58/58), done.


## Installing Dependencies

In [3]:
!pip install transformers==4.42.0 \
             accelerate==0.27.2 \
             tiktoken==0.7.0 \
             einops==0.7.0 \
             transformers_stream_generator==0.0.4 \
             peft==0.11.1 \
             trl==0.9.6 \
             Jinja2==3.1.4 \
             bitsandbytes \
             "numpy<=2.0.0 "\
             pyomo
# Install the Linux system-level math solvers that Pyomo relies on
!apt-get install -y coinor-cbc glpk-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
glpk-utils is already the newest version (5.0-1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


## Load Necessary modules

In [4]:
import os
import json
import sys


import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    pipeline
)

import peft
import trl
from pyomo.environ import *
import pyomo.environ as aml

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Current GPU: Tesla T4


In [5]:
# Load the necessary prompt modules from the official LLOMPT repo to build prompts
module_path = os.path.abspath('LLMOPT/prompts')
print(f"module_path: {module_path}")
# Add to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

import generate_prompt
import self_correction_prompt

module_path: /content/LLMOPT/prompts


## Load the Model
Use quantization to compress the model

In [6]:
# Configure the 4-bit quantization layout to protect cloud VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "ant-opt/LLMOPT-Qwen2.5-14B"

# Download and cache the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Download, quantize, and load the 14B model safely
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True        # Forces shard-by-shard loading directly into RAM/VRAM
)

print("done")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

model-00011-of-00012.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00012-of-00012.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

done


## Load the nl4opt Dataset

In [7]:
dataset_path = "LLMOPT/data/testset/nl4opt_test.jsonl"

# open and laod the dataset (json file)
with open(dataset_path, "r") as f:
    test_problems = [json.loads(line) for line in f]

print(f"Successfully loaded {len(test_problems)} test items from NL4Opt.")
print(f"Sample test problem:\n{json.dumps(test_problems[0], indent=1)}")

Successfully loaded 230 test items from NL4Opt.
Sample test problem:
{
 "question": "There has been an oil spill in the ocean and ducks need to be taken to shore to be cleaned either by boat or by canoe. A boat can take 10 ducks per trip while a canoe can take 8 ducks per trip. Since the boats are motor powered, they take 20 minutes per trip while the canoes take 40 minutes per trip. In order to avoid further environmental damage, there can be at most 12 boat trips and at least 60% of the trips should be by canoe. If at least 300 ducks need to be taken to shore, how many of each transportation method should be used to minimize the total amount of time needed to transport the ducks?",
 "answer": 1160,
 "ori": "5_nl4opt_test",
 "index": 1
}


## Testing and Evaluating the Model

### Copy the Necessary Functions from the official LLOMPT REPO And Create New functions to do the Self Correction Loop

In [10]:
import subprocess
# inference to get five elements
def infer_five_elem(question):
    messages = [
        {"role": "user", "content": generate_prompt.Q2F(question)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None


# inference to get pyomo python code
def infer_code(five_elem):
    messages = [
        {"role": "user", "content": generate_prompt.F2C(five_elem)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')


# execute the code
def test_code(code_str):
    code_path = f"./test.py"
    with open(code_path, "w") as f1:
        f1.write(code_str)

    ans = subprocess.run(f"python {code_path}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    # return answer logs, error code
    return str(ans.stdout.decode('gbk', errors='ignore')), str(ans.stderr.decode('gbk', errors='ignore'))

In [11]:
# this one is not from the official repo
def self_correct(ques, five, code, output, error):
    messages = [
        {"role": "user", "content": self_correction_prompt.self_correction(ques, five, code, output, error)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans


def self_correct_analysis_five_elem_prompt(analysis, ques, five):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The problem is as follows.

{ques}

The five-element formulation is as follows.

{five}

Based on the analysis. Please write the corresponding five-element model. Please use LaTeX and ``` plain text environment to complete the following template to model the above optimization problem into five elements: 

```
## Sets: 
[You need to fill in]

## Parameters: 
[You need to fill in]

## Variables: 
[You need to fill in]

## Objective: 
[You need to fill in]

## Constraints: 
[You need to fill in]
```
"""

    
def self_correct_analysis_code_prompt(analysis, five, code):
    return f"""For the following optimization problem, modeling is performed, and pyomo code is generated and executed based on the modeling. Please fix the modeling and code based on the analysis.

The analysis is as follows. 

{analysis}

The five-element formulation is as follows.

{five}

The code is as follows.

{code}

Based on the analysis. Please write the corresponding Pyomo code. Please add `from pyomo.environ import *` at the beginning of your code (You can add other `import` as well). Please print the optimal solution and the value of the objective function. Please do not output the running log. You need to write it in the form of a class and add a main function: 

```python
[write your code here]
```
"""

def self_correct_analysis_five_elem(analysis, ques, five):
    messages = [
        {"role": "user", "content": self_correct_analysis_five_elem_prompt(analysis, ques, five)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    response = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")

    if "```text" in response:
        return response.split("```text")[1].split("```")[0]
    elif "```plaintext" in response:
        return response.split("```plaintext")[1].split("```")[0]
    elif "```" in response:
        return response.split("```")[1].split("```")[0]
    else:
        return None

def self_correct_analysis_code(analysis, five, code):
    messages = [
        {"role": "user", "content": self_correct_analysis_code_prompt(analysis, five, code)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=8192
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ans = response.replace("\\\\n", "\n").replace("&#39;","'").replace("&lt;", "<").replace("&gt;", ">").replace("\\\\\"","\"")
    return ans.split("```python")[1].split("```")[0].replace('print("\\\\\n', 'print("').replace('print(f"\\\\\n', 'print(f"')

### Running the model

In [12]:
TEST_LIMIT = 10  # Evaluate 10 test problems
MAX_CORRECTION_ATTEMPTS = 3 
START_INDEX = 20
END_INDEX = TEST_LIMIT + START_INDEX

print(f"Launching real-time evaluation over {TEST_LIMIT} problems...")

real_time_data = test_problems[START_INDEX:END_INDEX]

successful_oneshot_count = 0
successful_corrected_count = 0
for i, item in enumerate(real_time_data):
    print(f"\nProcessing Problem {i+1}/{TEST_LIMIT}...")
    problem_description = item['question']
    problem_answer = str(item['answer'])
    print('question = ' + problem_description)
    print("answer = " + problem_answer)
    # --- PHASE 1: FIVE-ELEMENT FORMULATION ---  
    five_element_text = infer_five_elem(problem_description)
    
    # --- PHASE 2: INITIAL CODE GENERATION ---
    code = infer_code(five_element_text)
   
    # --- PHASE 3: EXECUTION & AUTO-TESTING SELF CORRECTION LOOP ---
    attempt = 0
    solved_successfully = False
    oneshot_win = False
    wrong_code = []

    while attempt < MAX_CORRECTION_ATTEMPTS:
        # Clean up any leftover markdown block syntax before execution
        run_res = test_code(code)

        print(f"attempt {attempt} output: {run_res[0]}")
        check_answer = problem_answer in run_res[0]

        if not run_res[1] and check_answer:
            solved_successfully = True
            if attempt == 0:
                oneshot_win = True
            break
        else:
            attempt += 1
            wrong_code.append(code)
            # Run Self-Correction Generation Pass
            correction_prompt_input = {
                'ques': problem_description, 
                'five': five_element_text, 
                'code': code, 
                'output': run_res[0], 
                'error': run_res[1]
            }
            
            self_correct_output = self_correct(**correction_prompt_input)
            five_element_text = self_correct_analysis_five_elem(self_correct_output, problem_description, five_element_text)
            code = self_correct_analysis_code(self_correct_output, five_element_text, code)

    if oneshot_win:
        successful_oneshot_count += 1
    if solved_successfully:
        successful_corrected_count += 1
    
    print(f"↳ Result - OneShot Win: {oneshot_win} | Final Success: {solved_successfully} (Attempts: {attempt})")
    if wrong_code:
        for i, wc in enumerate(wrong_code):
            print("================================================================")
            print(f"Attempt {i}:\n\n{wc}")
            print("================================================================")

# --- RESULTS SUMMARY (Indentation Fixed) ---
print("\n==================== ACTUAL TASK 1 METRICS ====================")
print(f"Total Problems Processed: {TEST_LIMIT}")
print(f"Accuracy WITHOUT Self-Correction: {(successful_oneshot_count/TEST_LIMIT)*100:.2f}%")
print(f"Accuracy WITH Self-Correction: {(successful_corrected_count/TEST_LIMIT)*100:.2f}%")
print("================================================================")

Launching real-time evaluation over 10 problems...

Processing Problem 1/10...
question = A bakery bakes bagels and croissants. A batch of bagels can be made using 2 hours of oven time and 0.25 hours of pastry chef time. A batch of croissants is more complicated, so while they take 1 hour of oven time, they take 2 hours of pastry chef time. In a day, the bakery has at most 70 hours available for the oven and 32 pastry chef hours available. Using all the available capacity, what is the maximum profit the bakery can generate assuming the profit per batch is $20 and $40 respectively for a batch of bagels and a batch of croissants.
answer = 1060
attempt 0 output: Status: ok
Termination Condition: optimal
Optimal Solution Found
Objective Value (Total Profit): 1060.0
Batch Production Levels:
  Bagels: 29.0
  Croissants: 12.0

↳ Result - OneShot Win: True | Final Success: True (Attempts: 0)

Processing Problem 2/10...
question = Platinum in combination with palladium has been used as a cataly

## Report

Total Problems Processed: 10
Accuracy WITHOUT Self-Correction: 70.00%
Accuracy WITH Self-Correction: 80.00%

Reasons for the delta: Used quantization/compression techniques so even though running the test directly on ant-opt/LLMOPT-Qwen2.5-14B which is already fine tuned for this specific task, the model im running should have less precision than the real fine-tuned model.  
And since I only test the model using a small portion of the dataset so the result is more likely to not be accurate. My self correction loop might not be as good as the ones the author made and I also set the max attempt to 3 instead of 12. 